![Week 4 Day 1](../assets/banners/w4d1.png)

# Week 4 Day 1 - The Agentic RAG Spectrum: Retrieve, Grade, Rewrite, Answer

Welcome to week 4! 🎉

Look how far GradeWise has come. In week 2 it got a **memory**: Emma's CS201 notes, chunked, embedded and searchable with hybrid retrieval, with a citation on every passage. In week 3 it learned to **check its own work**: a LangGraph workflow that extracts the exam, validates it, retries when something is wrong and pauses for Emma's approval.

This week brings those two ideas together, and then GradeWise learns a brand-new skill: **reading handwriting**.

Today is the first half of that story. Our week 2 RAG pipeline was a straight line: retrieve, then answer, whatever came back. That's fine when the search goes well. But what happens when a student asks *"why does my program freeze forever?"* and the notes only ever say *deadlock*? Today you'll build RAG that **notices a bad search, rewrites the question and tries again**: agentic RAG.

![Week 4 roadmap](../assets/roadmap/week4.png)

### 🔁 Where we are
- **Week 2:** the knowledge base, `get_retriever()` (dense, BM25, hybrid) and `answer_question()` with citations
- **Week 3:** agents, `StateGraph`, conditional edges, loops, `Send` and human-in-the-loop
- **This week:** agentic RAG + evidence for grading (Days 1–2), then reading handwritten answer sheets (Days 3–5)

### 🎯 By the end of this lab you will
1. 🗺️ Place **Adaptive**, **Corrective** and **Self-RAG** on the agentic RAG spectrum, and know what each adds
2. 🧪 Build a tiny corrective-RAG loop from scratch: retrieve → grade → rewrite → answer, with a retry cap
3. 🧱 Read GradeWise's `build_agentic_rag()`, draw its graph and watch it run, **offline**, with a scripted model
4. 🚀 Run it for real with your own model

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | Day 1 - Week 4 Kickoff: Recap and Roadmap | 🎬 |
| 2 | Day 1 - The Agentic RAG Spectrum: Adaptive, Corrective and Self-RAG | 🎬 |
| 3 | Day 1 - Building the Agentic RAG Graph: Retrieve, Grade, Rewrite, Answer | 🧪 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Run the cells <b>in order</b>, top to bottom, with the <code>.venv</code> kernel. Good news: almost everything today runs <b>offline and free</b>. We use a keyword-only (BM25) copy of the knowledge base and a "scripted" fake model, so you can watch the whole graph work before spending a cent. Only Part 6 calls a real model.</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Most of today's cells still run offline; the few that call a model will")
        print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

In [ ]:
from gradewise.config import OUTPUT_DIR, settings
from gradewise.console import success

success(
    "Your settings",
    {"Everyday model": settings.chat_model, "Vision model (Days 3-5)": settings.transcription_model},
    next_step="Part 1: the agentic RAG spectrum",
)

## Part 1: Week 4 kickoff, and why "retrieve then answer" isn't enough

Here's the week 2 pipeline, the one inside `gradewise/rag/answer.py`:

![Diagram](../assets/diagrams/46158717d331.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  Q[Question] --> R[Retrieve top-k] --> G[Generate with citations] --> A[Answer]
```

</details>


It has one big weakness: **it trusts the retriever blindly.** If the search returns the wrong chunks, the model either answers from the wrong notes or says "the notes don't cover that" when they actually do. Students rarely use the textbook's words: they say *freeze* for deadlock, *the handshake thing* for TCP, *the table splitting* for normalisation.

The fix is to give the pipeline a little judgement: let it look at what came back and decide what to do next. That is all "agentic RAG" means: **an LLM makes decisions about retrieval**, instead of a fixed script.

## Part 2: The agentic RAG spectrum

"Agentic RAG" isn't one design; it's a spectrum, from "one extra decision" to "the model critiques every sentence". Three well-known patterns mark the way:

### 🧭 Adaptive RAG: decide *whether* (and how) to retrieve
A router looks at the question first. Small talk or general knowledge? Answer directly. A simple fact? One retrieval. A hard multi-part question? Several retrievals.

![Diagram](../assets/diagrams/b71f29df3be8.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  Q[Question] --> Router{Router}
  Router -- "no retrieval needed" --> Direct[Answer directly]
  Router -- "simple" --> One[Retrieve once] --> A1[Answer]
  Router -- "complex" --> Multi[Retrieve, step by step] --> A2[Answer]
```

</details>


### 🔧 Corrective RAG (CRAG): *grade* what came back, and fix bad searches
After retrieving, a grader asks "are these chunks actually relevant?". If not, the system rewrites the query (or falls back to another source, like web search) and tries again.

![Diagram](../assets/diagrams/21e52f495540.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  Q[Question] --> R[Retrieve] --> G{Relevant?}
  G -- yes --> A[Answer]
  G -- no --> W[Rewrite query] --> R
```

</details>


### 🪞 Self-RAG: the model *reflects* on its own answer
The model decides when to retrieve, then critiques its own draft: is each passage relevant? Is every claim **supported** by a passage? Is the answer useful? Unsupported drafts are regenerated.

![Diagram](../assets/diagrams/32afbb1e13ef.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  Q[Question] --> R[Retrieve] --> D[Draft answer] --> C{Supported by<br/>the passages?}
  C -- yes --> A[Answer]
  C -- no --> D
```

</details>


### Putting them side by side

| Pattern | Extra decision the LLM makes | Fixes | Cost |
|---|---|---|---|
| 2-step RAG (week 2) | none | – | 1 LLM call |
| **Adaptive** | *Do I need to retrieve at all?* | wasted retrieval on small talk | +1 small call |
| **Corrective** | *Are these chunks relevant? If not, rewrite.* | bad searches, vocabulary mismatch | +1 grade call, +rewrites |
| **Self-RAG** | *Is my answer supported and useful?* | hallucinated claims | +1 or more critique calls |

💡 Each step to the right buys reliability with **more LLM calls**: more latency and more cost. The art is to add exactly the decisions your problem needs. GradeWise's study assistant combines the first two: a router (adaptive) and a grade-and-rewrite loop (corrective). Tomorrow we add the Self-RAG idea: checking that answers are **grounded**.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Before Part 3: our loop will rewrite the question whenever the grader says "not relevant". What happens if a student asks something the course notes simply <b>don't cover</b>, like "Who won the 1966 World Cup?" Predict how the loop behaves, and what we must add to stop it.</span>
        </td>
    </tr>
</table>

## Part 3: A corrective RAG loop from scratch

Before we open GradeWise's version, let's build the idea in about 40 lines, with **no LLM at all**: a keyword "retriever" over four tiny notes, a rule-based grader, and a rewrite step that swaps student words for course words. Plain Python makes the *shape* of the loop easy to see.

First, the tiny knowledge base and the search:

In [ ]:
import re

NOTES = {
    "1.4 CPU scheduling": "The scheduler picks which ready process runs next. Round robin gives each one a time slice.",
    "1.6 Deadlocks": "A deadlock is a set of processes each waiting for a resource held by another. "
    "Four conditions: mutual exclusion, hold and wait, no preemption, circular wait.",
    "2.5 Transport layer": "TCP gives reliable, ordered delivery. UDP is connectionless and faster.",
    "3.4 Transactions": "ACID: atomicity, consistency, isolation and durability.",
}
STOPWORDS = {
    "a",
    "an",
    "the",
    "is",
    "are",
    "my",
    "why",
    "does",
    "do",
    "what",
    "when",
    "who",
    "won",
    "of",
    "and",
    "each",
}


def words(text: str) -> set[str]:
    return {w.rstrip("s") for w in re.findall(r"[a-z0-9]+", text.lower())} - STOPWORDS


def search(query: str, k: int = 1) -> list[str]:
    """Keyword overlap: a (very) poor man's BM25."""
    scored = sorted(((len(words(query) & words(text)), title) for title, text in NOTES.items()), reverse=True)
    return [title for score, title in scored[:k] if score > 0]


print(search("What are the four conditions for deadlock?"))
print(search("Why does my program freeze forever?"))  # 😬 student words, not course words

The second search finds **nothing**, even though the deadlock note is exactly what the student needs. Now the graph. We reuse everything from week 3: a `TypedDict` state, nodes that return partial updates, and a **conditional edge** that decides where to go next.

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph

MAX_REWRITES = 2  # ⚠️ the answer to the Stop & think: without a cap, an unanswerable question loops forever
GLOSSARY = {"freeze": "deadlock", "stuck": "deadlock", "hang": "deadlock", "handshake": "TCP", "fast": "UDP"}


class MiniState(TypedDict):
    question: str
    query: str
    docs: list[str]
    rewrites: int
    answer: str


def retrieve(state: MiniState) -> dict:
    return {"docs": search(state["query"])}


def grade(state: MiniState) -> Literal["generate", "rewrite"]:
    """A conditional edge, not a node: it only decides where to go next."""
    if state["docs"] or state["rewrites"] >= MAX_REWRITES:
        return "generate"
    return "rewrite"


def rewrite(state: MiniState) -> dict:
    query = " ".join(GLOSSARY.get(w.rstrip("?.,!").lower(), w) for w in state["query"].split())
    return {"query": query, "rewrites": state["rewrites"] + 1}


def generate(state: MiniState) -> dict:
    if not state["docs"]:
        return {"answer": "The course notes don't cover that."}
    title = state["docs"][0]
    return {"answer": f"{NOTES[title]} [{title}]"}


mini = StateGraph(MiniState)
mini.add_node("retrieve", retrieve)
mini.add_node("rewrite", rewrite)
mini.add_node("generate", generate)
mini.add_edge(START, "retrieve")
mini.add_conditional_edges("retrieve", grade)
mini.add_edge("rewrite", "retrieve")
mini.add_edge("generate", END)
mini_rag = mini.compile()

In [ ]:
def ask(question: str) -> None:
    state = mini_rag.invoke({"question": question, "query": question, "docs": [], "rewrites": 0, "answer": ""})
    print(f"❓ {question}\n   final query: {state['query']!r} · rewrites: {state['rewrites']}")
    print(f"   💬 {state['answer']}\n")


ask("What are the four conditions for deadlock?")
ask("Why does my program freeze forever?")
ask("Who won the 1966 World Cup?")

Three questions, three paths through the same graph:
1. **Good search** → straight to the answer, zero rewrites.
2. **Vocabulary mismatch** → the grader says no, the rewrite turns *freeze* into *deadlock*, the second search hits.
3. **Not in the notes** → it rewrites up to the cap, then answers honestly: *"The course notes don't cover that."*

Let's look at the shape LangGraph built. `draw_mermaid()` gives us the diagram as text; no internet needed.

In [ ]:
from gradewise.display import show_mermaid

mermaid = mini_rag.get_graph().draw_mermaid()
show_mermaid(mermaid)

In [ ]:
# 💡 JupyterLab 4.1+ and recent VS Code render ```mermaid blocks in Markdown. Elsewhere, paste the
#    text above into https://mermaid.live
display(Markdown(f"```mermaid\n{mermaid}\n```"))

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Notice the dashed arrows out of <code>retrieve</code>: that's the conditional edge. LangGraph worked out the possible targets from the <code>Literal["generate", "rewrite"]</code> return type of <code>grade</code>. Forget the type hint and the diagram (and LangGraph Studio) can't show where the edge goes. And never ship a loop without a cap: an LLM grader that keeps saying "no" would otherwise burn tokens until it hits LangGraph's recursion limit.</span>
        </td>
    </tr>
</table>

## Part 4: 🧱 GradeWise's agentic RAG graph

Our toy has a rule-based grader and a glossary. The real one, in `gradewise/rag/agentic_rag.py`, replaces both with an LLM, and follows the official LangGraph agentic RAG tutorial:

| Node / edge | What it does |
|---|---|
| `generate_query_or_respond` | The **adaptive** part: the model either answers directly (small talk) or calls the `retrieve_course_notes` tool with a focused search query |
| `retrieve` | A `ToolNode` that runs the tool on our week 2 retriever |
| `grade_documents` | The **corrective** part: a conditional edge that asks the model for a structured `GradeDocuments(binary_score="yes"/"no")` |
| `rewrite_question` | Asks the model for ONE better search query, adds it as a new message, and loops back |
| `generate_answer` | Answers the **original** question from the latest notes, with citations |

Open `gradewise/rag/agentic_rag.py` in the editor and find these three details:
1. `@tool(response_format="content_and_artifact")`: the model sees the formatted text, our code keeps the real `Document` objects for citations.
2. `MAX_REWRITES = 2`, checked at the top of `grade_documents`, exactly like our toy.
3. The grading prompt wraps the notes in `<notes>` tags and says they are *data, not instructions*. Retrieved text is untrusted input.

### Drawing it, offline

`build_agentic_rag()` accepts any model and any retriever, so we can **build and draw** the graph with a placeholder model. Nothing is called until we invoke it.

In [ ]:
from gradewise.display import show_mermaid

from langchain_core.language_models.fake_chat_models import FakeListChatModel

from gradewise.rag import build_agentic_rag


class NoRetriever:
    """Anything with .invoke(query) works as a retriever for building the graph."""

    def invoke(self, query: str) -> list:
        return []


preview = build_agentic_rag(llm=FakeListChatModel(responses=["-"]), retriever=NoRetriever())
agentic_mermaid = preview.get_graph().draw_mermaid()
show_mermaid(agentic_mermaid)

In [ ]:
display(Markdown(f"```mermaid\n{agentic_mermaid}\n```"))

Compare it with our toy: same loop, plus one extra entry decision (`generate_query_or_respond` can go straight to `__end__`). The edge labelled `tools` comes from LangGraph's prebuilt `tools_condition`: "did the model ask for a tool?"

## Part 5: Watch it run, for free

To see every step without an API key, we need two things:

1. **A real retriever that works offline.** We ingest the CS201 PDFs into a separate Chroma collection using `DeterministicFakeEmbedding`. Fake embeddings are random, so dense search is meaningless, but **BM25 is pure keyword maths**: it works perfectly. (It takes a few seconds the first time; after that the ingest is skipped, as you learned in week 2.)
2. **A scripted model.** A fake chat model that replays the messages we give it, in order: a tool call, a "no" grade, a rewrite, and so on. It's the same trick `tests/test_rag.py` uses.

In [ ]:
from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise.ingestion import ingest
from gradewise.rag import get_retriever

OFFLINE_DB = OUTPUT_DIR / "week4_offline_db"  # 💡 a separate store: never mixes fake vectors with your real ones
fake_embeddings = DeterministicFakeEmbedding(size=64)

# 📦 ingest() reads datasets.kb_files(): all 4 PDFs, or unit1 + the handbook with DATA_SIZE=small
report = ingest(collection="cs201_offline", embeddings=fake_embeddings, persist_dir=OFFLINE_DB, quiet=True)
bm25 = get_retriever("cs201_offline", k=3, mode="bm25", embeddings=fake_embeddings, persist_dir=OFFLINE_DB)
print(f"{report.chunks_total} chunks in the offline store")

for doc in bm25.invoke("why do processes get stuck waiting"):
    print("•", doc.metadata["source"], "§", doc.metadata["section"])

Look at that: the student-style query *"why do processes get stuck waiting"* pulls in the learning outcomes, **CPU scheduling** and processes, but not a word about deadlocks. A perfect case for the grade-and-rewrite loop. Now the scripted model:

In [ ]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage


class ScriptedToolModel(GenericFakeChatModel):
    """Replays scripted messages, and 'supports' tools by ignoring them."""

    def bind_tools(self, tools, **kwargs):
        return self


def tool_call(name: str, args: dict, call_id: str) -> AIMessage:
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": call_id, "type": "tool_call"}])


script = [
    tool_call("retrieve_course_notes", {"query": "why do processes get stuck waiting"}, "1"),  # route → search
    tool_call("GradeDocuments", {"binary_score": "no"}, "2"),  # grade: scheduling notes ≠ deadlock
    AIMessage("deadlock four necessary conditions circular wait"),  # rewrite
    tool_call("retrieve_course_notes", {"query": "deadlock four necessary conditions circular wait"}, "3"),
    tool_call("GradeDocuments", {"binary_score": "yes"}, "4"),  # grade: relevant now
    AIMessage("Processes deadlock when each holds a resource another needs [unit1_operating_systems.pdf p4]."),
]
graph = build_agentic_rag(llm=ScriptedToolModel(messages=iter(script)), retriever=bm25)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">The script contains 6 messages. Before running the next cell, predict the order in which the nodes will run, and how many times <code>retrieve</code> appears. (Hint: <code>grade_documents</code> is an edge, so it won't show up as a step of its own.)</span>
        </td>
    </tr>
</table>

In [ ]:
# 💡 stream_mode="updates" yields {node_name: what_that_node_returned} after each step (week 3, Day 4)
question = "Why do my processes get stuck waiting for each other?"
for update in graph.stream({"messages": [HumanMessage(question)], "rewrites": 0}, stream_mode="updates"):
    for node, output in update.items():
        message = output["messages"][-1]
        if getattr(message, "tool_calls", None):
            detail = f"search → {message.tool_calls[0]['args']['query']!r}"
        elif getattr(message, "artifact", None) is not None:
            detail = "found → " + "; ".join(d.metadata["section"] for d in message.artifact)
        else:
            detail = message.text[:90]
        print(f"▶ {node:<26} {detail}")

That is the full corrective loop on the **real** CS201 notes: the first search found scheduling material, the grade said "no", the rewrite used the course's own vocabulary, and the second search landed in §1.6 Deadlocks.

`run_agentic_rag()` wraps the same thing and hands back `(answer, docs, state)`. Here it is with a fresh script, for the question the notes can't answer, with `max_rewrites=1`:

In [ ]:
from gradewise.rag import run_agentic_rag, unique_citations

script = [
    tool_call("retrieve_course_notes", {"query": "1966 World Cup winner"}, "1"),
    tool_call("GradeDocuments", {"binary_score": "no"}, "2"),
    AIMessage("1966 football World Cup final"),
    tool_call("retrieve_course_notes", {"query": "1966 football World Cup final"}, "3"),
    # 💡 the cap is reached: grade_documents goes straight to generate_answer, no grading call
    AIMessage("The course notes don't cover that."),
]
answer, docs, state = run_agentic_rag(
    "Who won the 1966 World Cup?", llm=ScriptedToolModel(messages=iter(script)), retriever=bm25, max_rewrites=1
)
print("💬", answer)
print("🔁 rewrites:", state["rewrites"], "· messages in state:", len(state["messages"]))
print("📚 last search returned:", unique_citations(docs) or "nothing")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">Students phrase questions in their own words, and that is exactly when a plain RAG bot fails them. A grade-and-rewrite loop is a cheap fix (one small extra call per question) that makes a study assistant feel like it <i>understands</i> the course. The same loop powers internal help-desks, policy assistants and support bots, where the user's vocabulary never matches the documentation's.</span>
        </td>
    </tr>
</table>

## Part 6: Run it for real

Now let the real model make every decision. We keep the free BM25 retriever, so only the chat model costs anything: with the default `gpt-6-luna`, about a tenth of a cent for both questions. If you have run `uv run week2/pro_implementation/ingest.py`, try `retriever=None` to use your real hybrid store instead.

No key? You'll see a friendly *Missing API key* panel. Everything above still stands.

In [ ]:
from gradewise.models import chat_model

real_graph = build_agentic_rag(llm=chat_model(temperature=0), retriever=bm25)
answer, docs, state = run_agentic_rag("Why do my processes get stuck waiting for each other?", graph=real_graph)
display(Markdown(answer))
print("rewrites:", state["rewrites"], "· sources:", unique_citations(docs))

In [ ]:
# 🧪 TRY: small talk should skip retrieval entirely (the adaptive router at work)
answer, docs, state = run_agentic_rag("Hi! Thanks for helping me revise.", graph=real_graph)
print(answer, "\n· retrieved chunks:", len(docs))

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Everything you ran today lives in <code>gradewise/rag/agentic_rag.py</code>, and <code>week4/project/agentic_rag.py</code> is a command-line demo of it. Try <code>uv run week4/project/agentic_rag.py --diagram</code> (free: it only prints the Mermaid graph). In week 5, the grading agent gets retrieval as a <b>tool</b>, so it can decide for itself when it needs more evidence.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 🗺️ placed **Adaptive** (route first), **Corrective** (grade and rewrite) and **Self-RAG** (critique the answer) on the agentic RAG spectrum
- 🧪 built a corrective loop from scratch, and saw why every loop needs a **rewrite cap**
- 🧱 drew GradeWise's `build_agentic_rag()` graph and watched it fix a bad search on the real CS201 notes, offline
- 🚀 ran it with a real model, and watched the router skip retrieval for small talk

**Tomorrow (Day 2):** answers are only as good as their evidence. We'll check that every claim is **grounded** in a cited passage, catch hallucinated citations, and build the **evidence retriever** the week 5 grader will use to mark Sam's deadlock answer.

⏭ **Next:** `week4/day2.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Make the toy graph <b>adaptive</b>. Add a <code>route</code> step before <code>retrieve</code> that sends greetings and thank-yous straight to a friendly reply, and everything else to the search. Then draw the graph again: you should see a new branch from <code>__start__</code>. Bonus: change <code>MAX_REWRITES</code> to 0 and explain what changes for "Why does my program freeze forever?". Solutions are in <code>week4/solutions/day1_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• LangGraph's agentic RAG tutorial: <a href="https://docs.langchain.com/oss/python/langgraph/agentic-rag">docs.langchain.com/oss/python/langgraph/agentic-rag</a><br/>• Corrective RAG (Yan et al., 2024): <a href="https://arxiv.org/abs/2401.15884">arxiv.org/abs/2401.15884</a><br/>• Self-RAG (Asai et al., 2023): <a href="https://arxiv.org/abs/2310.11511">arxiv.org/abs/2310.11511</a><br/>• Adaptive-RAG (Jeong et al., 2024): <a href="https://arxiv.org/abs/2403.14403">arxiv.org/abs/2403.14403</a></span>
        </td>
    </tr>
</table>